In [1]:
import os
import warnings
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIG
# =============================================================================
HRV_1MIN_PATH = "/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/Wearable-Device-Dataset/HRV/Wearable_v2_HRV_1min_discard.csv"
HRV_2MIN_PATH = "/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/Wearable-Device-Dataset/HRV/Wearable_v2_HRV_2min_discard.csv"
TPV_PATH = "/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/Wearable-Device-Dataset/TPV/Wearable_v2_BVP_TPV_noQC.csv"

OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/Wearable-Device-Dataset/cls_results_top5"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.30

LABEL_COL = "status"
SUBJECT_COL = "subject"

CLASSIFIER_ORDER = ["DT", "LR", "RF", "SVC"]

# HRV는 교수님 말씀 기준 의미 있는 5개만 사용
HRV_FEATURE_COLS = [
    "RMSSD",
    "SDNN",
    "LF",
    "HF",
    "LF_HF",
]

# 방금 boxplot + ANOVA + effect size에서 뽑힌 TPV top-5
TPV_FEATURE_COLS = [
    "TPV17",
    "TPV12",
    "TPV26",
    "TPV22",
    "TPV3",
]


# =============================================================================
# Data loading
# =============================================================================
def read_csv_auto(path):
    with open(path, "r", encoding="utf-8-sig") as f:
        first_line = f.readline()
    sep = "\t" if first_line.count("\t") > 3 else ","
    df = pd.read_csv(path, sep=sep)
    df.columns = df.columns.str.strip()
    return df


def prepare_df(path, dataset_name, candidate_features):
    df = read_csv_auto(path)

    df = df[df[LABEL_COL].isin([0, 1])].copy()
    df["label_major"] = df[LABEL_COL].astype(int)

    feat_cols = [c for c in candidate_features if c in df.columns]

    for c in feat_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    before = len(df)
    df = df.dropna(subset=feat_cols + ["label_major", SUBJECT_COL]).copy()
    after = len(df)

    print(f"\n[{dataset_name}]")
    print(f"Loaded       : {before}")
    print(f"After dropna : {after}")
    print(f"Class dist   : {df['label_major'].value_counts().to_dict()}")
    print(f"Subjects     : {sorted(df[SUBJECT_COL].unique())}")
    print(f"Used features: {len(feat_cols)}")
    print(feat_cols)

    return df, feat_cols


# =============================================================================
# Classifiers
# =============================================================================
def get_classifiers():
    return {
        "DT": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "LR": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        "RF": RandomForestClassifier(
            n_estimators=100,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "SVC": SVC(
            kernel="rbf",
            probability=True,
            random_state=RANDOM_STATE
        ),
    }


# =============================================================================
# Evaluation
# =============================================================================
def evaluate(clf, X_train, X_test, y_train, y_test):
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_train)
    X_te = scaler.transform(X_test)

    clf.fit(X_tr, y_train)
    y_pred = clf.predict(X_te)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="binary", zero_division=0)

    try:
        y_prob = clf.predict_proba(X_te)[:, 1]
        auc = roc_auc_score(y_test, y_prob) if len(np.unique(y_test)) == 2 else np.nan
    except Exception:
        auc = np.nan

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

    return acc, f1, auc, cm


def run_loso(df, feat_cols, feature_name):
    results = []
    subjects = sorted(df[SUBJECT_COL].unique())

    print(f"\n{'=' * 80}")
    print(f"LOSO Binary Classification — {feature_name}")
    print(f"{'=' * 80}")

    for test_subj in subjects:
        df_train = df[df[SUBJECT_COL] != test_subj].copy()
        df_test = df[df[SUBJECT_COL] == test_subj].copy()

        df_train = df_train.dropna(subset=feat_cols + ["label_major"])
        df_test = df_test.dropna(subset=feat_cols + ["label_major"])

        X_train = df_train[feat_cols].values.astype(float)
        y_train = df_train["label_major"].values.astype(int)

        X_test = df_test[feat_cols].values.astype(float)
        y_test = df_test["label_major"].values.astype(int)

        if len(df_test) < 2 or len(np.unique(y_train)) < 2:
            print(f"[Test={test_subj}] 데이터 부족 → SKIP")
            continue

        train_dist = dict(zip(*np.unique(y_train, return_counts=True)))
        test_dist = dict(zip(*np.unique(y_test, return_counts=True)))

        print(
            f"\n── Test Subject: {test_subj} | "
            f"Train={len(X_train)} {train_dist}, Test={len(X_test)} {test_dist} ──"
        )

        for clf_name, clf in get_classifiers().items():
            acc, f1, auc, cm = evaluate(clf, X_train, X_test, y_train, y_test)

            results.append({
                "Evaluation": "LOSO",
                "Feature Set": feature_name,
                "Subject": test_subj,
                "Classifier": clf_name,
                "Accuracy": acc,
                "Binary-F1": f1,
                "AUC": auc,
                "TN": cm[0, 0],
                "FP": cm[0, 1],
                "FN": cm[1, 0],
                "TP": cm[1, 1],
                "N_train": len(X_train),
                "N_test": len(X_test),
            })

            print(
                f"{clf_name:4s} | Acc={acc:.4f} | F1={f1:.4f} | "
                f"AUC={auc:.4f} | CM=[TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}]"
            )

    return pd.DataFrame(results)


def run_personalized(df, feat_cols, feature_name):
    results = []
    subjects = sorted(df[SUBJECT_COL].unique())

    print(f"\n{'=' * 80}")
    print(f"Personalized Binary Classification — {feature_name}")
    print(f"{'=' * 80}")

    for subj in subjects:
        df_subj = df[df[SUBJECT_COL] == subj].copy()
        df_subj = df_subj.dropna(subset=feat_cols + ["label_major"])

        X = df_subj[feat_cols].values.astype(float)
        y = df_subj["label_major"].values.astype(int)

        n_classes = len(np.unique(y))

        if len(df_subj) < 5 or n_classes < 2:
            print(f"[{subj}] 샘플 부족 (n={len(df_subj)}, classes={n_classes}) → SKIP")
            continue

        class_dist = dict(zip(*np.unique(y, return_counts=True)))

        if min(class_dist.values()) < 2:
            print(f"[{subj}] minority class 부족 {class_dist} → SKIP")
            continue

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE,
            stratify=y,
        )

        print(
            f"\n── {subj} | Train={len(X_train)}, Test={len(X_test)}, "
            f"Class dist={class_dist} ──"
        )

        for clf_name, clf in get_classifiers().items():
            acc, f1, auc, cm = evaluate(clf, X_train, X_test, y_train, y_test)

            results.append({
                "Evaluation": "Personalized",
                "Feature Set": feature_name,
                "Subject": subj,
                "Classifier": clf_name,
                "Accuracy": acc,
                "Binary-F1": f1,
                "AUC": auc,
                "TN": cm[0, 0],
                "FP": cm[0, 1],
                "FN": cm[1, 0],
                "TP": cm[1, 1],
                "N": len(df_subj),
                "N_train": len(X_train),
                "N_test": len(X_test),
            })

            print(
                f"{clf_name:4s} | Acc={acc:.4f} | F1={f1:.4f} | "
                f"AUC={auc:.4f} | CM={cm.tolist()}"
            )

    return pd.DataFrame(results)


# =============================================================================
# Summary
# =============================================================================
def format_mean_std(series):
    mean = series.mean()
    std = series.std()

    if pd.isna(mean):
        return ""

    if pd.isna(std):
        std = 0.0

    return f"{mean:.4f}±{std:.4f}"


def make_summary_table(df_res, eval_name):
    feature_order = ["HRV 1-min", "HRV 2-min", "TPV Top-5"]
    rows = []

    for feature in feature_order:
        for clf in CLASSIFIER_ORDER:
            sub = df_res[
                (df_res["Evaluation"] == eval_name)
                & (df_res["Feature Set"] == feature)
                & (df_res["Classifier"] == clf)
            ]

            rows.append({
                "Feature Set": feature,
                "Classifier": clf,
                "Accuracy": format_mean_std(sub["Accuracy"]) if not sub.empty else "",
                "Binary-F1": format_mean_std(sub["Binary-F1"]) if not sub.empty else "",
                "AUC": format_mean_std(sub["AUC"]) if not sub.empty else "",
            })

    return pd.DataFrame(rows)


def save_outputs(df_all):
    raw_path = os.path.join(OUTPUT_DIR, "Wearable_Device_Dataset_top5_all_results_raw.csv")
    loso_path = os.path.join(OUTPUT_DIR, "Wearable_Device_Dataset_top5_LOSO_summary.csv")
    per_path = os.path.join(OUTPUT_DIR, "Wearable_Device_Dataset_top5_Personalized_summary.csv")
    xlsx_path = os.path.join(OUTPUT_DIR, "Wearable_Device_Dataset_top5_Final_Summary.xlsx")

    df_all.to_csv(raw_path, index=False, encoding="utf-8-sig")

    loso_table = make_summary_table(df_all, "LOSO")
    per_table = make_summary_table(df_all, "Personalized")

    loso_table.to_csv(loso_path, index=False, encoding="utf-8-sig")
    per_table.to_csv(per_path, index=False, encoding="utf-8-sig")

    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        loso_table.to_excel(writer, index=False, sheet_name="LOSO")
        per_table.to_excel(writer, index=False, sheet_name="Personalized")
        df_all.to_excel(writer, index=False, sheet_name="Raw_Results")

        for sheet_name in ["LOSO", "Personalized", "Raw_Results"]:
            ws = writer.book[sheet_name]

            for col in ["A", "B", "C", "D", "E"]:
                ws.column_dimensions[col].width = 18

            for row in ws.iter_rows():
                for cell in row:
                    cell.alignment = cell.alignment.copy(horizontal="center", vertical="center")

    print("\nSaved:")
    print(raw_path)
    print(loso_path)
    print(per_path)
    print(xlsx_path)

    print("\n1) Cross Subject (LOSO) Binary Performance")
    print(loso_table.to_string(index=False))

    print("\n2) Personalized Binary Performance")
    print(per_table.to_string(index=False))


# =============================================================================
# Main
# =============================================================================
if __name__ == "__main__":
    configs = [
        ("HRV 1-min", HRV_1MIN_PATH, HRV_FEATURE_COLS),
        ("HRV 2-min", HRV_2MIN_PATH, HRV_FEATURE_COLS),
        ("TPV Top-5", TPV_PATH, TPV_FEATURE_COLS),
    ]

    all_results = []

    for feature_name, path, feature_cols in configs:
        df, used_feat_cols = prepare_df(path, feature_name, feature_cols)

        print(f"\n[{feature_name}] 최종 사용 피처 ({len(used_feat_cols)}개)")
        print(used_feat_cols)

        loso_res = run_loso(df, used_feat_cols, feature_name)
        per_res = run_personalized(df, used_feat_cols, feature_name)

        all_results.append(loso_res)
        all_results.append(per_res)

    df_all = pd.concat(all_results, axis=0, ignore_index=True)
    save_outputs(df_all)

    print("\nDone.")


[HRV 1-min]
Loaded       : 1085
After dropna : 1083
Class dist   : {0: 882, 1: 201}
Subjects     : ['f01', 'f02', 'f03', 'f04', 'f05', 'f06', 'f07', 'f08', 'f09', 'f10', 'f11', 'f12', 'f13', 'f15', 'f16', 'f17', 'f18']
Used features: 5
['RMSSD', 'SDNN', 'LF', 'HF', 'LF_HF']

[HRV 1-min] 최종 사용 피처 (5개)
['RMSSD', 'SDNN', 'LF', 'HF', 'LF_HF']

LOSO Binary Classification — HRV 1-min

── Test Subject: f01 | Train=1001 {np.int64(0): np.int64(818), np.int64(1): np.int64(183)}, Test=82 {np.int64(0): np.int64(64), np.int64(1): np.int64(18)} ──
DT   | Acc=0.7073 | F1=0.2941 | AUC=0.5530 | CM=[TN=53 FP=11 FN=13 TP=5]
LR   | Acc=0.7805 | F1=0.0000 | AUC=0.6215 | CM=[TN=64 FP=0 FN=18 TP=0]
RF   | Acc=0.7805 | F1=0.0000 | AUC=0.5399 | CM=[TN=64 FP=0 FN=18 TP=0]
SVC  | Acc=0.7805 | F1=0.0000 | AUC=0.6293 | CM=[TN=64 FP=0 FN=18 TP=0]

── Test Subject: f02 | Train=1031 {np.int64(0): np.int64(844), np.int64(1): np.int64(187)}, Test=52 {np.int64(0): np.int64(38), np.int64(1): np.int64(14)} ──
DT   | Acc=

In [1]:
import os
import warnings
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIG
# =============================================================================
HRV_1MIN_PATH = "/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/Wearable-Device-Dataset/HRV/Wearable_v2_HRV_1min_discard.csv"
HRV_2MIN_PATH = "/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/Wearable-Device-Dataset/HRV/Wearable_v2_HRV_2min_discard.csv"
TPV_PATH = "/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/Wearable-Device-Dataset/TPV/Wearable_v2_BVP_TPV_noQC.csv"

OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/Wearable-Device-Dataset/cls_results_top5"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.30

LABEL_COL = "status"
SUBJECT_COL = "subject"

CLASSIFIER_ORDER = ["DT", "LR", "RF", "SVC"]

# HRV는 교수님 말씀 기준 의미 있는 5개만 사용
HRV_FEATURE_COLS = [
    "RMSSD",
    "SDNN",
    "LF",
    "HF",
    "LF_HF",
]

# 방금 boxplot + ANOVA + effect size에서 뽑힌 TPV top-5
TPV_FEATURE_COLS = [
    "TPV17",
    "TPV12",
    "TPV26",
    "TPV22",
    "TPV3",
]


# =============================================================================
# Data loading
# =============================================================================
def read_csv_auto(path):
    with open(path, "r", encoding="utf-8-sig") as f:
        first_line = f.readline()
    sep = "\t" if first_line.count("\t") > 3 else ","
    df = pd.read_csv(path, sep=sep)
    df.columns = df.columns.str.strip()
    return df


def prepare_df(path, dataset_name, candidate_features):
    df = read_csv_auto(path)

    df = df[df[LABEL_COL].isin([0, 1])].copy()
    df["label_major"] = df[LABEL_COL].astype(int)

    feat_cols = [c for c in candidate_features if c in df.columns]

    for c in feat_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    before = len(df)
    df = df.dropna(subset=feat_cols + ["label_major", SUBJECT_COL]).copy()
    after = len(df)

    print(f"\n[{dataset_name}]")
    print(f"Loaded       : {before}")
    print(f"After dropna : {after}")
    print(f"Class dist   : {df['label_major'].value_counts().to_dict()}")
    print(f"Subjects     : {sorted(df[SUBJECT_COL].unique())}")
    print(f"Used features: {len(feat_cols)}")
    print(feat_cols)

    return df, feat_cols


# =============================================================================
# Classifiers
# =============================================================================
def get_classifiers():
    return {
        "DT": DecisionTreeClassifier(
            random_state=RANDOM_STATE,
            class_weight="balanced"        # 추가
        ),
        "LR": LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE,
            class_weight="balanced"        # 추가
        ),
        "RF": RandomForestClassifier(
            n_estimators=100,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced"        # 추가
        ),
        "SVC": SVC(
            kernel="rbf",
            probability=True,
            random_state=RANDOM_STATE,
            class_weight="balanced"        # 추가
        ),
    }


# =============================================================================
# Evaluation
# =============================================================================
def evaluate(clf, X_train, X_test, y_train, y_test):
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_train)
    X_te = scaler.transform(X_test)

    clf.fit(X_tr, y_train)
    y_pred = clf.predict(X_te)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="binary", zero_division=0)

    try:
        y_prob = clf.predict_proba(X_te)[:, 1]
        auc = roc_auc_score(y_test, y_prob) if len(np.unique(y_test)) == 2 else np.nan
    except Exception:
        auc = np.nan

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

    return acc, f1, auc, cm


def run_loso(df, feat_cols, feature_name):
    results = []
    subjects = sorted(df[SUBJECT_COL].unique())

    print(f"\n{'=' * 80}")
    print(f"LOSO Binary Classification — {feature_name}")
    print(f"{'=' * 80}")

    for test_subj in subjects:
        df_train = df[df[SUBJECT_COL] != test_subj].copy()
        df_test = df[df[SUBJECT_COL] == test_subj].copy()

        df_train = df_train.dropna(subset=feat_cols + ["label_major"])
        df_test = df_test.dropna(subset=feat_cols + ["label_major"])

        X_train = df_train[feat_cols].values.astype(float)
        y_train = df_train["label_major"].values.astype(int)

        X_test = df_test[feat_cols].values.astype(float)
        y_test = df_test["label_major"].values.astype(int)

        if len(df_test) < 2 or len(np.unique(y_train)) < 2:
            print(f"[Test={test_subj}] 데이터 부족 → SKIP")
            continue

        train_dist = dict(zip(*np.unique(y_train, return_counts=True)))
        test_dist = dict(zip(*np.unique(y_test, return_counts=True)))

        print(
            f"\n── Test Subject: {test_subj} | "
            f"Train={len(X_train)} {train_dist}, Test={len(X_test)} {test_dist} ──"
        )

        for clf_name, clf in get_classifiers().items():
            acc, f1, auc, cm = evaluate(clf, X_train, X_test, y_train, y_test)

            results.append({
                "Evaluation": "LOSO",
                "Feature Set": feature_name,
                "Subject": test_subj,
                "Classifier": clf_name,
                "Accuracy": acc,
                "Binary-F1": f1,
                "AUC": auc,
                "TN": cm[0, 0],
                "FP": cm[0, 1],
                "FN": cm[1, 0],
                "TP": cm[1, 1],
                "N_train": len(X_train),
                "N_test": len(X_test),
            })

            print(
                f"{clf_name:4s} | Acc={acc:.4f} | F1={f1:.4f} | "
                f"AUC={auc:.4f} | CM=[TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}]"
            )

    return pd.DataFrame(results)


def run_personalized(df, feat_cols, feature_name):
    results = []
    subjects = sorted(df[SUBJECT_COL].unique())

    print(f"\n{'=' * 80}")
    print(f"Personalized Binary Classification — {feature_name}")
    print(f"{'=' * 80}")

    for subj in subjects:
        df_subj = df[df[SUBJECT_COL] == subj].copy()
        df_subj = df_subj.dropna(subset=feat_cols + ["label_major"])

        X = df_subj[feat_cols].values.astype(float)
        y = df_subj["label_major"].values.astype(int)

        n_classes = len(np.unique(y))

        if len(df_subj) < 5 or n_classes < 2:
            print(f"[{subj}] 샘플 부족 (n={len(df_subj)}, classes={n_classes}) → SKIP")
            continue

        class_dist = dict(zip(*np.unique(y, return_counts=True)))

        if min(class_dist.values()) < 2:
            print(f"[{subj}] minority class 부족 {class_dist} → SKIP")
            continue

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE,
            stratify=y,
        )

        print(
            f"\n── {subj} | Train={len(X_train)}, Test={len(X_test)}, "
            f"Class dist={class_dist} ──"
        )

        for clf_name, clf in get_classifiers().items():
            acc, f1, auc, cm = evaluate(clf, X_train, X_test, y_train, y_test)

            results.append({
                "Evaluation": "Personalized",
                "Feature Set": feature_name,
                "Subject": subj,
                "Classifier": clf_name,
                "Accuracy": acc,
                "Binary-F1": f1,
                "AUC": auc,
                "TN": cm[0, 0],
                "FP": cm[0, 1],
                "FN": cm[1, 0],
                "TP": cm[1, 1],
                "N": len(df_subj),
                "N_train": len(X_train),
                "N_test": len(X_test),
            })

            print(
                f"{clf_name:4s} | Acc={acc:.4f} | F1={f1:.4f} | "
                f"AUC={auc:.4f} | CM={cm.tolist()}"
            )

    return pd.DataFrame(results)


# =============================================================================
# Summary
# =============================================================================
def format_mean_std(series):
    mean = series.mean()
    std = series.std()

    if pd.isna(mean):
        return ""

    if pd.isna(std):
        std = 0.0

    return f"{mean:.4f}±{std:.4f}"


def make_summary_table(df_res, eval_name):
    feature_order = ["HRV 1-min", "HRV 2-min", "TPV Top-5"]
    rows = []

    for feature in feature_order:
        for clf in CLASSIFIER_ORDER:
            sub = df_res[
                (df_res["Evaluation"] == eval_name)
                & (df_res["Feature Set"] == feature)
                & (df_res["Classifier"] == clf)
            ]

            rows.append({
                "Feature Set": feature,
                "Classifier": clf,
                "Accuracy": format_mean_std(sub["Accuracy"]) if not sub.empty else "",
                "Binary-F1": format_mean_std(sub["Binary-F1"]) if not sub.empty else "",
                "AUC": format_mean_std(sub["AUC"]) if not sub.empty else "",
            })

    return pd.DataFrame(rows)


def save_outputs(df_all):
    raw_path = os.path.join(OUTPUT_DIR, "Wearable_Device_Dataset_top5_all_results_raw.csv")
    loso_path = os.path.join(OUTPUT_DIR, "Wearable_Device_Dataset_top5_LOSO_summary.csv")
    per_path = os.path.join(OUTPUT_DIR, "Wearable_Device_Dataset_top5_Personalized_summary.csv")
    xlsx_path = os.path.join(OUTPUT_DIR, "Wearable_Device_Dataset_top5_Final_Summary.xlsx")

    df_all.to_csv(raw_path, index=False, encoding="utf-8-sig")

    loso_table = make_summary_table(df_all, "LOSO")
    per_table = make_summary_table(df_all, "Personalized")

    loso_table.to_csv(loso_path, index=False, encoding="utf-8-sig")
    per_table.to_csv(per_path, index=False, encoding="utf-8-sig")

    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        loso_table.to_excel(writer, index=False, sheet_name="LOSO")
        per_table.to_excel(writer, index=False, sheet_name="Personalized")
        df_all.to_excel(writer, index=False, sheet_name="Raw_Results")

        for sheet_name in ["LOSO", "Personalized", "Raw_Results"]:
            ws = writer.book[sheet_name]

            for col in ["A", "B", "C", "D", "E"]:
                ws.column_dimensions[col].width = 18

            for row in ws.iter_rows():
                for cell in row:
                    cell.alignment = cell.alignment.copy(horizontal="center", vertical="center")

    print("\nSaved:")
    print(raw_path)
    print(loso_path)
    print(per_path)
    print(xlsx_path)

    print("\n1) Cross Subject (LOSO) Binary Performance")
    print(loso_table.to_string(index=False))

    print("\n2) Personalized Binary Performance")
    print(per_table.to_string(index=False))


# =============================================================================
# Main
# =============================================================================
if __name__ == "__main__":
    configs = [
        ("HRV 1-min", HRV_1MIN_PATH, HRV_FEATURE_COLS),
        ("HRV 2-min", HRV_2MIN_PATH, HRV_FEATURE_COLS),
        ("TPV Top-5", TPV_PATH, TPV_FEATURE_COLS),
    ]

    all_results = []

    for feature_name, path, feature_cols in configs:
        df, used_feat_cols = prepare_df(path, feature_name, feature_cols)

        print(f"\n[{feature_name}] 최종 사용 피처 ({len(used_feat_cols)}개)")
        print(used_feat_cols)

        loso_res = run_loso(df, used_feat_cols, feature_name)
        per_res = run_personalized(df, used_feat_cols, feature_name)

        all_results.append(loso_res)
        all_results.append(per_res)

    df_all = pd.concat(all_results, axis=0, ignore_index=True)
    save_outputs(df_all)

    print("\nDone.")


[HRV 1-min]
Loaded       : 1085
After dropna : 1083
Class dist   : {0: 882, 1: 201}
Subjects     : ['f01', 'f02', 'f03', 'f04', 'f05', 'f06', 'f07', 'f08', 'f09', 'f10', 'f11', 'f12', 'f13', 'f15', 'f16', 'f17', 'f18']
Used features: 5
['RMSSD', 'SDNN', 'LF', 'HF', 'LF_HF']

[HRV 1-min] 최종 사용 피처 (5개)
['RMSSD', 'SDNN', 'LF', 'HF', 'LF_HF']

LOSO Binary Classification — HRV 1-min

── Test Subject: f01 | Train=1001 {np.int64(0): np.int64(818), np.int64(1): np.int64(183)}, Test=82 {np.int64(0): np.int64(64), np.int64(1): np.int64(18)} ──
DT   | Acc=0.6829 | F1=0.2778 | AUC=0.5373 | CM=[TN=51 FP=13 FN=13 TP=5]
LR   | Acc=0.6341 | F1=0.2857 | AUC=0.6163 | CM=[TN=46 FP=18 FN=12 TP=6]
RF   | Acc=0.7561 | F1=0.0000 | AUC=0.5668 | CM=[TN=62 FP=2 FN=18 TP=0]
SVC  | Acc=0.6829 | F1=0.4091 | AUC=0.6102 | CM=[TN=47 FP=17 FN=9 TP=9]

── Test Subject: f02 | Train=1031 {np.int64(0): np.int64(844), np.int64(1): np.int64(187)}, Test=52 {np.int64(0): np.int64(38), np.int64(1): np.int64(14)} ──
DT   | Acc